# 00 · 환경과 작은 ZIP 묶음 점검

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** Colab GPU와 GCS 접근, ZIP 크기·SHA-256·CRC·이미지/JSON 짝을 먼저 점검합니다.
- **입력:** Training의 H_0007 원천·라벨 ZIP 한 쌍, 본인 GCP 프로젝트와 비공개 버킷
- **출력:** 환경 정보, ZIP·라벨 검사 기록, 성공 시 smoke 결과 파일
- **실제 저장된 실행 상태:** GPU·GCS·ZIP CRC·안전한 압축 해제는 PASS입니다. 이미지 내용 검증은 잘린 JPEG 1개를 발견해 AssertionError로 중단했습니다. 전체 smoke 성공 기록은 아닙니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/00_environment_smoke.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/00_environment_smoke.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 고등부 예선 — GCS·Colab 환경·데이터 스모크 테스트

이 노트북의 목적은 모델 학습 전에 환경과 데이터가 정상인지 작은 학습 묶음 하나로 검증하는 것입니다.

- 최초 테스트 대상: Training의 H_0007 한 쌍(이미지 ZIP + JSON ZIP)
- 성공 기준: GPU 인식(선택), ZIP CRC, 이미지/JSON 2,839쌍, JSON 파싱, 이미지 디코딩, 라벨 시각화가 모두 통과
- 공식 Validation 폴더의 VS_*.zip, VL_*.zip은 예선 분석에 사용하거나 업로드하지 않습니다.
- Google Drive는 사용하지 않습니다. 원본 ZIP과 결과물은 비공개 GCS 버킷에 보존하고 현재 작업할 한 쌍만 Colab 로컬 디스크로 내려받습니다.
- 무료 체험 크레딧은 사용료를 대신 지불하는 크레딧입니다. Cloud Console에서 잔액과 남은 날짜를 주기적으로 확인하십시오.

중요: 아래 셀은 위에서 아래로 한 번씩 실행합니다. 셀 왼쪽의 재생 버튼을 누르거나 Shift+Enter를 사용하십시오.


## 0. 먼저 GCS에 테스트 ZIP 2개 올리기

1. 별도 제공된 DCC_GCS_무료체험_설정_업로드_가이드.md에 따라 GCP 프로젝트와 비공개 Standard 버킷을 만듭니다.
2. Windows Google Cloud CLI에서 TS_H_0007.zip을 gs://버킷/dcc/raw/source/에 올립니다.
3. TL_H_0007.zip을 gs://버킷/dcc/raw/labels/에 올립니다.

원천 이미지 ZIP:

    YOUR_LOCAL_DATA_ROOT/Training/01.원천데이터/TS_H_0007.zip

라벨 JSON ZIP:

    YOUR_LOCAL_DATA_ROOT/Training/02.라벨링데이터/TL_H_0007.zip

GCS 객체 경로 dcc/raw/source와 dcc/raw/labels는 분석 전략 문서 Mission 1-5의 구조와 동일합니다. Colab 브라우저 직접 업로드는 사용하지 않습니다.

GPU 확인을 하려면 Colab 메뉴에서 런타임 → 런타임 유형 변경 → 하드웨어 가속기 GPU를 선택합니다. 특정 GPU 모델은 보장되지 않습니다. 업로드·압축 해제만 오래 수행할 때는 CPU 런타임을 쓰면 컴퓨팅 단위를 아낄 수 있습니다.


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
ram_kib = int(next(line.split()[1] for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')))
print(f'RAM: {ram_kib / 1024**2:.1f} GiB')
disk = shutil.disk_usage('/content')
print(f'Local disk: total={disk.total/1024**3:.1f} GiB, free={disk.free/1024**3:.1f} GiB')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'GPU VRAM: {props.total_memory/1024**3:.1f} GiB')
    a = torch.randn((2048, 2048), device='cuda')
    torch.cuda.synchronize()
    started = time.perf_counter()
    _ = a @ a
    torch.cuda.synchronize()
    print(f'GPU matrix smoke test: PASS ({time.perf_counter()-started:.3f}s)')
    del a, _
    torch.cuda.empty_cache()
else:
    print('GPU test: SKIP — 필요하면 런타임 유형을 GPU로 바꾼 뒤 이 셀을 다시 실행하세요.')

nvidia = shutil.which('nvidia-smi')
if nvidia:
    result = subprocess.run([nvidia, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True, check=False)
    print('nvidia-smi:', result.stdout.strip())

print('\n주의: GPU 인식 성공은 현재 가속기 할당 확인이지, 결제 상태 자체를 증명하는 검사는 아닙니다.')


Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
RAM: 12.7 GiB
Local disk: total=235.7 GiB, free=188.5 GiB
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU VRAM: 14.6 GiB
GPU matrix smoke test: PASS (0.242s)
nvidia-smi: Tesla T4, 15360 MiB, 580.82.07

주의: GPU 인식 성공은 현재 가속기 할당 확인이지, 결제 상태 자체를 증명하는 검사는 아닙니다.


In [ ]:
import json
import math

from google.colab import auth

# 아래 두 값만 Cloud Console에 표시된 실제 값으로 바꾸세요.
PROJECT_ID = 'YOUR_GCP_PROJECT_ID'
BUCKET_NAME = 'YOUR_GCS_BUCKET'

if PROJECT_ID.startswith('CHANGE_ME') or BUCKET_NAME.startswith('CHANGE_ME'):
    raise ValueError('PROJECT_ID와 BUCKET_NAME을 실제 값으로 먼저 바꾸세요.')
if BUCKET_NAME.startswith('gs://'):
    raise ValueError('BUCKET_NAME에는 gs://를 빼고 버킷 이름만 입력하세요.')

auth.authenticate_user(project_id=PROJECT_ID)

def run_gcloud(arguments, capture_output=True):
    command = ['gcloud', *[str(item) for item in arguments]]
    result = subprocess.run(command, text=True, capture_output=capture_output, check=False)
    if result.returncode != 0:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        raise subprocess.CalledProcessError(result.returncode, command)
    return result

run_gcloud(['config', 'set', 'project', PROJECT_ID, '--quiet'])
active_account = run_gcloud(['auth', 'list', '--filter=status:ACTIVE', '--format=value(account)']).stdout.strip()
print('Authenticated account:', active_account)
print('GCP project:', PROJECT_ID)

GCS_BUCKET = f'gs://{BUCKET_NAME}'
GCS_ROOT = f'{GCS_BUCKET}/dcc'
GCS_SOURCE_DIR = f'{GCS_ROOT}/raw/source'
GCS_LABEL_DIR = f'{GCS_ROOT}/raw/labels'
GCS_ARTIFACT_DIR = f'{GCS_ROOT}/notebook/smoke_outputs'
SOURCE_ZIP_GCS = f'{GCS_SOURCE_DIR}/TS_H_0007.zip'
LABEL_ZIP_GCS = f'{GCS_LABEL_DIR}/TL_H_0007.zip'

EXPECTED_SMOKE = {
    'TS_H_0007.zip': {
        'bytes': 766_466_626,
        'sha256': 'd1de7f2c3ca4cecf07c45414dcc6ccb1013f405bbed5151c3cf3953203409e3f',
    },
    'TL_H_0007.zip': {
        'bytes': 2_339_421,
        'sha256': '840fcb27f15d5d6446aeda867d68b30be460b3157e05399cad1f3e161f400ae9',
    },
}

LOCAL_ROOT = Path('/content/dcc')
LOCAL_ZIP_DIR = LOCAL_ROOT / 'runtime' / 'smoke_zips'
LOCAL_STAGE_DIR = LOCAL_ROOT / 'stage' / 'H_0007'
LOCAL_IMAGE_DIR = LOCAL_STAGE_DIR / 'images'
LOCAL_LABEL_DIR = LOCAL_STAGE_DIR / 'json'
LOCAL_ARTIFACT_DIR = LOCAL_ROOT / 'runtime' / 'smoke_artifacts'

for folder in [LOCAL_ZIP_DIR, LOCAL_IMAGE_DIR, LOCAL_LABEL_DIR, LOCAL_ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

bucket_metadata = json.loads(run_gcloud(['storage', 'buckets', 'describe', GCS_BUCKET, '--format=json']).stdout)
uniform_enabled = bucket_metadata.get('uniform_bucket_level_access', False)
public_access_prevention = bucket_metadata.get('public_access_prevention')
assert uniform_enabled is True, 'Uniform bucket-level access가 활성화되지 않았습니다.'
assert public_access_prevention == 'enforced', 'Public access prevention이 enforced가 아닙니다.'
print('Bucket location:', bucket_metadata.get('location'))
print('Bucket storage class:', bucket_metadata.get('storage_class'))

def describe_gcs_object(uri):
    return json.loads(run_gcloud(['storage', 'objects', 'describe', uri, '--format=json']).stdout)

for uri in [SOURCE_ZIP_GCS, LABEL_ZIP_GCS]:
    name = uri.rsplit('/', 1)[-1]
    metadata = describe_gcs_object(uri)
    actual_bytes = int(metadata['size'])
    expected_bytes = EXPECTED_SMOKE[name]['bytes']
    if actual_bytes != expected_bytes:
        raise IOError(f'GCS 객체 크기 불일치: {name} — actual={actual_bytes:,}, expected={expected_bytes:,}')
    print(f'FOUND {name}: {actual_bytes/1024**3:.3f} GiB')
print('GCS bucket and object exact-size check: PASS')


Authenticated account: YOUR_ACCOUNT@example.invalid
GCP project: YOUR_GCP_PROJECT_ID
Bucket location: US-CENTRAL1
Bucket storage class: None
FOUND TS_H_0007.zip: 0.714 GiB
FOUND TL_H_0007.zip: 0.002 GiB
GCS bucket and object exact-size check: PASS


In [ ]:
import hashlib
import zipfile

zip_bytes = EXPECTED_SMOKE['TS_H_0007.zip']['bytes'] + EXPECTED_SMOKE['TL_H_0007.zip']['bytes']
expected_uncompressed_bytes = 767_746_903 + 5_964_059
required_bytes = int(zip_bytes + expected_uncompressed_bytes * 1.20 + 1024**3)
free_bytes = shutil.disk_usage('/content').free
print(f'ZIP total: {zip_bytes/1024**3:.3f} GiB')
print(f'Uncompressed estimate: {expected_uncompressed_bytes/1024**3:.3f} GiB')
print(f'Conservative local-space requirement: {required_bytes/1024**3:.3f} GiB')
print(f'Current local free space: {free_bytes/1024**3:.3f} GiB')
if free_bytes < required_bytes:
    raise RuntimeError('Colab 로컬 디스크가 부족합니다. 런타임을 재시작하거나 더 작은 작업만 남겨 두세요.')

def sha256_file(path, chunk_size=16 * 1024**2):
    digest = hashlib.sha256()
    with Path(path).open('rb') as reader:
        for block in iter(lambda: reader.read(chunk_size), b''):
            digest.update(block)
    return digest.hexdigest()

def download_gcs_verified(uri, dst, expected_bytes, expected_sha256):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_file() and dst.stat().st_size == expected_bytes:
        local_sha256 = sha256_file(dst)
        if local_sha256 == expected_sha256:
            print(f'SKIP download (verified SHA-256): {dst.name}')
            return local_sha256
    temp = dst.with_name(dst.name + '.part')
    if temp.exists():
        temp.unlink()
    print(f'Downloading {uri} ...')
    run_gcloud(['storage', 'cp', uri, str(temp)], capture_output=False)
    if temp.stat().st_size != expected_bytes:
        temp.unlink(missing_ok=True)
        raise IOError(f'GCS 다운로드 크기 불일치: {dst.name}')
    downloaded_sha256 = sha256_file(temp)
    if downloaded_sha256 != expected_sha256:
        temp.unlink(missing_ok=True)
        raise IOError(f'GCS 다운로드 SHA-256 불일치: {dst.name}. 객체를 다시 업로드하세요.')
    os.replace(temp, dst)
    print(f'SHA-256 PASS: {dst.name} = {downloaded_sha256}')
    return downloaded_sha256

SOURCE_ZIP_LOCAL = LOCAL_ZIP_DIR / 'TS_H_0007.zip'
LABEL_ZIP_LOCAL = LOCAL_ZIP_DIR / 'TL_H_0007.zip'
SOURCE_SHA256 = download_gcs_verified(SOURCE_ZIP_GCS, SOURCE_ZIP_LOCAL, EXPECTED_SMOKE[SOURCE_ZIP_LOCAL.name]['bytes'], EXPECTED_SMOKE[SOURCE_ZIP_LOCAL.name]['sha256'])
LABEL_SHA256 = download_gcs_verified(LABEL_ZIP_GCS, LABEL_ZIP_LOCAL, EXPECTED_SMOKE[LABEL_ZIP_LOCAL.name]['bytes'], EXPECTED_SMOKE[LABEL_ZIP_LOCAL.name]['sha256'])

def zip_uncompressed_bytes(path):
    with zipfile.ZipFile(path) as zf:
        return sum(info.file_size for info in zf.infolist() if not info.is_dir())

actual_uncompressed_bytes = zip_uncompressed_bytes(SOURCE_ZIP_LOCAL) + zip_uncompressed_bytes(LABEL_ZIP_LOCAL)
assert actual_uncompressed_bytes == expected_uncompressed_bytes
print('GCS → Colab local ZIP download and SHA-256: PASS')


ZIP total: 0.716 GiB
Uncompressed estimate: 0.721 GiB
Conservative local-space requirement: 2.581 GiB
Current local free space: 188.467 GiB
SHA-256 PASS: TS_H_0007.zip = d1de7f2c3ca4cecf07c45414dcc6ccb1013f405bbed5151c3cf3953203409e3f
SHA-256 PASS: TL_H_0007.zip = 840fcb27f15d5d6446aeda867d68b30be460b3157e05399cad1f3e161f400ae9
GCS → Colab local ZIP download and SHA-256: PASS


GCS에 있는 ZIP을 Colab 로컬 디스크로 내려받은 뒤 압축을 풉니다. GCS 원본은 그대로 보존하고 Colab에는 현재 작업할 한 쌍만 둡니다. Colab 런타임이 종료되면 /content 아래 파일은 사라지므로 결과 보고서는 마지막에 GCS로 다시 올립니다.


In [ ]:
from collections import Counter
import stat

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def normalized_member_name(raw_name):
    if '\x00' in raw_name or '\\' in raw_name or raw_name.startswith('//'):
        raise ValueError(f'안전하지 않은 ZIP 경로: {raw_name!r}')
    candidate = raw_name[1:] if raw_name.startswith('/') else raw_name
    if not candidate or '/' in candidate or candidate in {'.', '..'} or ':' in candidate:
        raise ValueError(f'중첩 또는 안전하지 않은 ZIP 경로: {raw_name!r}')
    return candidate

def inspect_zip(path, allowed_suffixes):
    names = []
    suffix_counts = Counter()
    with zipfile.ZipFile(path) as zf:
        for info in zf.infolist():
            if info.is_dir():
                raise ValueError(f'예상하지 않은 ZIP 디렉터리: {info.filename!r}')
            if info.flag_bits & 0x1:
                raise ValueError(f'암호화된 ZIP member: {info.filename!r}')
            mode = (info.external_attr >> 16) & 0xFFFF
            if stat.S_IFMT(mode) not in {0, stat.S_IFREG}:
                raise ValueError(f'symlink 또는 special ZIP member: {info.filename!r}')
            name = normalized_member_name(info.filename)
            suffix = Path(name).suffix.lower()
            suffix_counts[suffix] += 1
            if suffix not in allowed_suffixes:
                raise ValueError(f'예상하지 않은 확장자: {info.filename!r}')
            names.append(name)
    folded_names = [name.casefold() for name in names]
    if len(folded_names) != len(set(folded_names)):
        duplicates = [name for name, count in Counter(names).items() if count > 1]
        raise ValueError(f'ZIP 내부 중복 파일명: {duplicates[:5]}')
    return names, suffix_counts

image_names, image_suffix_counts = inspect_zip(SOURCE_ZIP_LOCAL, IMAGE_SUFFIXES)
label_names, label_suffix_counts = inspect_zip(LABEL_ZIP_LOCAL, {'.json'})
image_stems = {Path(name).stem for name in image_names}
label_stems = {Path(name).stem for name in label_names}

print('Image ZIP suffixes:', dict(image_suffix_counts))
print('Label ZIP suffixes:', dict(label_suffix_counts))
print('Image count:', len(image_names))
print('Label count:', len(label_names))
assert len(image_names) == 2839, f'예상 이미지 2,839개와 다름: {len(image_names)}'
assert len(label_names) == 2839, f'예상 JSON 2,839개와 다름: {len(label_names)}'
assert image_stems == label_stems, f'짝 불일치 — image only={len(image_stems-label_stems)}, label only={len(label_stems-image_stems)}'

for path in [SOURCE_ZIP_LOCAL, LABEL_ZIP_LOCAL]:
    print(f'CRC testing {path.name} ...')
    with zipfile.ZipFile(path) as zf:
        bad_member = zf.testzip()
    assert bad_member is None, f'CRC 오류 파일: {bad_member}'
print('ZIP structure, pair set, and CRC: PASS')


Image ZIP suffixes: {'.jpg': 2839}
Label ZIP suffixes: {'.json': 2839}
Image count: 2839
Label count: 2839
CRC testing TS_H_0007.zip ...
CRC testing TL_H_0007.zip ...
ZIP structure, pair set, and CRC: PASS


In [ ]:
def safe_flat_extract(zip_path, destination, allowed_suffixes, chunk_size=8 * 1024**2):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    destination_resolved = destination.resolve()
    seen = set()
    extracted = 0
    skipped = 0
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            name = normalized_member_name(info.filename)
            if Path(name).suffix.lower() not in allowed_suffixes:
                continue
            if name in seen:
                raise ValueError(f'중복 파일명: {name}')
            seen.add(name)
            output = (destination / name).resolve()
            if output.parent != destination_resolved:
                raise ValueError(f'대상 폴더를 벗어나는 ZIP 경로: {info.filename!r}')
            if output.is_file() and output.stat().st_size == info.file_size:
                skipped += 1
                continue
            temp = output.with_name(output.name + '.part')
            if temp.exists():
                temp.unlink()
            with zf.open(info, 'r') as reader, temp.open('wb') as writer:
                shutil.copyfileobj(reader, writer, length=chunk_size)
            if temp.stat().st_size != info.file_size:
                raise IOError(f'압축 해제 크기 불일치: {name}')
            os.replace(temp, output)
            extracted += 1
    print(f'{Path(zip_path).name}: extracted={extracted}, reused={skipped}')

safe_flat_extract(SOURCE_ZIP_LOCAL, LOCAL_IMAGE_DIR, IMAGE_SUFFIXES)
safe_flat_extract(LABEL_ZIP_LOCAL, LOCAL_LABEL_DIR, {'.json'})
print('Safe extraction: PASS')


TS_H_0007.zip: extracted=2839, reused=0
TL_H_0007.zip: extracted=2839, reused=0
Safe extraction: PASS


### 특정 단일 파일만 압축 해제 및 불러오기

전체 압축을 해제하지 않고, ZIP 파일 내에서 특정 이미지 파일 하나만 지정하여 `/content/` 폴더에 직접 압축을 해제하고 시각화하는 코드입니다.

In [ ]:
import json

import pandas as pd
from PIL import Image

image_paths = sorted(p for p in LOCAL_IMAGE_DIR.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)
label_paths = sorted(LOCAL_LABEL_DIR.glob('*.json'))
image_by_stem = {p.stem: p for p in image_paths}
label_by_stem = {p.stem: p for p in label_paths}

assert len(image_by_stem) == len(image_paths) == 2839
assert len(label_by_stem) == len(label_paths) == 2839
assert set(image_by_stem) == set(label_by_stem)

bad_images = []
image_sizes = {}
for path in image_paths:
    try:
        with Image.open(path) as image:
            image.load()
            image_sizes[path.name] = image.size
    except Exception as exc:
        bad_images.append((path.name, repr(exc)))
assert not bad_images, f'열리지 않는 이미지: {bad_images[:5]}'

rows = []
for label_path in label_paths:
    with label_path.open('r', encoding='utf-8-sig') as reader:
        record = json.load(reader)
    if not isinstance(record, dict):
        raise TypeError(f'JSON root가 object가 아님: {label_path.name}')
    image_info = record['image']
    annotations = record.get('annotations', [])
    categories = record.get('categories', [])
    if not isinstance(image_info, dict):
        raise TypeError(f'image가 object가 아님: {label_path.name}')
    if not isinstance(annotations, list):
        raise TypeError(f'annotations가 list가 아님: {label_path.name}')
    if not isinstance(categories, list) or not categories:
        raise TypeError(f'categories가 비어 있거나 list가 아님: {label_path.name}')
    image_name = str(image_info['filename'])
    if '/' in image_name or '\\' in image_name or Path(image_name).name != image_name:
        raise ValueError(f'JSON filename에 경로가 포함됨: {label_path.name} → {image_name!r}')
    image_path = LOCAL_IMAGE_DIR / image_name
    if not image_path.is_file():
        raise FileNotFoundError(f'JSON이 가리키는 이미지 없음: {label_path.name} → {image_name}')
    if image_path.stem != label_path.stem:
        raise ValueError(f'파일 stem 불일치: {image_path.name}, {label_path.name}')
    declared_size = (int(image_info['width']), int(image_info['height']))
    if image_sizes[image_name] != declared_size:
        raise ValueError(f'이미지 크기 불일치: {image_name} — actual={image_sizes[image_name]}, JSON={declared_size}')
    waterlevel_m = float(image_info['waterlevel'])
    if not math.isfinite(waterlevel_m):
        raise ValueError(f'유한하지 않은 waterlevel: {label_path.name}')
    for annotation in annotations:
        if not isinstance(annotation, dict):
            raise TypeError(f'annotation이 object가 아님: {label_path.name}')
        coordinates = annotation.get('relative_coordinates')
        if isinstance(coordinates, str):
            coordinates = json.loads(coordinates)
        if not isinstance(coordinates, dict):
            raise TypeError(f'relative_coordinates 파싱 실패: {label_path.name}')
        values = {}
        for key in ['center_x', 'center_y', 'width', 'height']:
            if key not in coordinates or not math.isfinite(float(coordinates[key])):
                raise ValueError(f'유효하지 않은 bbox {key}: {label_path.name}')
            values[key] = float(coordinates[key])
        if not (0 <= values['center_x'] <= 1 and 0 <= values['center_y'] <= 1):
            raise ValueError(f'bbox 중심 좌표 범위 오류: {label_path.name}')
        if not (0 < values['width'] <= 1 and 0 < values['height'] <= 1):
            raise ValueError(f'bbox 크기 범위 오류: {label_path.name}')
    category = categories[0] if categories else {}
    if category.get('mokjaType') != 'H':
        raise ValueError(f'mokjaType이 H가 아님: {label_path.name}')
    names = [str(item.get('name', '')) for item in annotations]
    rows.append({
        'stem': label_path.stem,
        'image_name': image_name,
        'label_name': label_path.name,
        'width': int(image_info['width']),
        'height': int(image_info['height']),
        'waterlevel_m': waterlevel_m,
        'mokja_type': category.get('mokjaType'),
        'clslevel': category.get('clslevel'),
        'annotation_count': len(annotations),
        'mokja_count': names.count('mokja'),
        'surface_count': names.count('surface'),
        'obstruction': json.dumps(image_info.get('obstruction', []), ensure_ascii=False),
    })

manifest = pd.DataFrame(rows).sort_values('stem').reset_index(drop=True)
assert len(manifest) == 2839
assert manifest['waterlevel_m'].notna().all()
assert (manifest['mokja_type'] == 'H').all()

manifest_path = LOCAL_ARTIFACT_DIR / 'smoke_manifest_H_0007.csv'
manifest.to_csv(manifest_path, index=False, encoding='utf-8-sig')

def upload_artifact(local_path):
    local_path = Path(local_path)
    remote_uri = f'{GCS_ARTIFACT_DIR}/{local_path.name}'
    run_gcloud(['storage', 'cp', str(local_path), remote_uri], capture_output=False)
    return remote_uri

manifest_uri = upload_artifact(manifest_path)

from datetime import datetime, timezone
smoke_result = {
    'status': 'PASS',
    'key': 'H_0007',
    'pair_count': int(len(manifest)),
    'source_size_bytes': int(SOURCE_ZIP_LOCAL.stat().st_size),
    'label_size_bytes': int(LABEL_ZIP_LOCAL.stat().st_size),
    'source_sha256': SOURCE_SHA256,
    'label_sha256': LABEL_SHA256,
    'waterlevel_min_m': float(manifest['waterlevel_m'].min()),
    'waterlevel_max_m': float(manifest['waterlevel_m'].max()),
    'finished_at_utc': datetime.now(timezone.utc).isoformat(),
}
result_path = LOCAL_ARTIFACT_DIR / 'smoke_result_H_0007.json'
result_path.write_text(json.dumps(smoke_result, ensure_ascii=False, indent=2), encoding='utf-8')
result_uri = upload_artifact(result_path)

print(f'Images decoded: {len(image_paths):,}')
print(f'JSON parsed and paired: {len(label_paths):,}')
print(f'Manifest saved: {manifest_uri}')
print(f'Result marker saved: {result_uri}')
print('Content validation: PASS')


AssertionError: 열리지 않는 이미지: [('H_0007_6aa9acd6-6fad-4e74-8ad7-a6b2ae29cefe.jpg', "OSError('image file is truncated (9 bytes not processed)')")]

In [ ]:
display(manifest.head(3))
display(manifest[['waterlevel_m', 'annotation_count', 'mokja_count', 'surface_count']].describe())
print('Water-level range:', manifest['waterlevel_m'].min(), '~', manifest['waterlevel_m'].max(), 'm')
print('Duplicate stems:', int(manifest['stem'].duplicated().sum()))
print('Missing water levels:', int(manifest['waterlevel_m'].isna().sum()))
print('Mokja type counts:')
display(manifest['mokja_type'].value_counts(dropna=False))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def parse_relative_coordinates(value):
    if isinstance(value, str):
        return json.loads(value)
    return value

ordered = manifest.sort_values('waterlevel_m').reset_index(drop=True)
indices = np.linspace(0, len(ordered) - 1, 4, dtype=int)
samples = ordered.iloc[indices]
colors = {'mokja': 'red', 'surface': 'deepskyblue'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (_, row) in zip(axes.ravel(), samples.iterrows()):
    image_path = LOCAL_IMAGE_DIR / row['image_name']
    label_path = LOCAL_LABEL_DIR / row['label_name']
    with label_path.open('r', encoding='utf-8-sig') as reader:
        record = json.load(reader)
    image = Image.open(image_path).convert('RGB')
    width, height = image.size
    ax.imshow(image)
    for annotation in record.get('annotations', []):
        name = str(annotation.get('name', ''))
        if name not in colors:
            continue
        coords = parse_relative_coordinates(annotation['relative_coordinates'])
        box_w = float(coords['width']) * width
        box_h = float(coords['height']) * height
        left = float(coords['center_x']) * width - box_w / 2
        top = float(coords['center_y']) * height - box_h / 2
        ax.add_patch(Rectangle((left, top), box_w, box_h, fill=False, linewidth=2, edgecolor=colors[name]))
        ax.text(left, max(0, top - 4), name, color='white', fontsize=9, backgroundcolor=colors[name])
    ax.set_title(f"{row['stem']} | waterlevel={row['waterlevel_m']:.2f} m")
    ax.axis('off')

fig.suptitle('H_0007 annotation smoke test — red: mokja, blue: surface', fontsize=14)
fig.tight_layout()
figure_path = LOCAL_ARTIFACT_DIR / 'smoke_labels_H_0007.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
figure_uri = upload_artifact(figure_path)
plt.show()
print(f'Figure saved: {figure_uri}')


## 1. H_0007이 전부 PASS한 뒤 전체 Training ZIP을 GCS에 올리기

Windows PowerShell에서 Training/02.라벨링데이터의 TL_*.zip 22개를 gs://버킷/dcc/raw/labels에 먼저 rsync합니다.

그다음 Training/01.원천데이터의 TS_*.zip 22개를 gs://버킷/dcc/raw/source에 rsync합니다. 명령은 별도 GCS 설정·업로드 가이드에 있습니다.

현재 로컬 데이터 기준 전체 Training 압축파일은 약 117.74 GiB입니다. 무료 체험 크레딧 잔액과 남은 날짜를 확인하고, 공식 Validation ZIP은 올리지 않습니다.

한 번에 전부 압축 해제하지 마십시오. 이후 전처리 노트북에서는 TS/TL 한 쌍씩 GCS에서 /content로 다운로드 → 검증 → 필요한 파생 데이터 생성 → 결과를 GCS에 업로드 → 로컬 임시 파일 정리 순서로 처리해야 합니다.


In [ ]:
EXPECTED_BY_KEY = {
    'A_0005': (6_691_689_410, 21_903_965),
    'A_0510': (14_906_742_900, 42_457_491),
    'A_1015': (11_165_900_539, 28_195_339),
    'B_0003': (7_715_251_257, 17_171_560),
    'B_0306': (5_661_978_328, 11_781_249),
    'B_0609': (2_589_522_515, 5_780_771),
    'C_0005': (9_087_926_484, 21_286_360),
    'C_0510': (19_338_857_630, 33_127_640),
    'C_1015': (15_482_673_854, 27_055_549),
    'E_0003': (5_019_586_536, 7_700_792),
    'E_0306': (2_762_769_824, 8_507_548),
    'E_0610': (2_753_727_777, 6_480_125),
    'F_0002': (1_567_936_700, 2_819_544),
    'F_0204': (1_862_771_404, 4_546_233),
    'G_0002': (2_265_269_014, 4_281_182),
    'G_0204': (3_137_701_657, 5_206_889),
    'H_0007': (766_466_626, 2_339_421),
    'H_0714': (1_442_602_803, 3_325_010),
    'H_1420': (1_278_235_340, 2_770_114),
    'S_0007': (2_055_352_703, 4_054_375),
    'S_0714': (3_803_208_130, 5_622_282),
    'S_1420': (4_792_296_782, 7_472_664),
}
EXPECTED_KEYS = set(EXPECTED_BY_KEY)
EXPECTED_SOURCE_SIZES = {f'TS_{key}.zip': sizes[0] for key, sizes in EXPECTED_BY_KEY.items()}
EXPECTED_LABEL_SIZES = {f'TL_{key}.zip': sizes[1] for key, sizes in EXPECTED_BY_KEY.items()}
EXPECTED_SOURCE_BYTES = 126_148_468_213
EXPECTED_LABEL_BYTES = 273_886_103
assert sum(EXPECTED_SOURCE_SIZES.values()) == EXPECTED_SOURCE_BYTES
assert sum(EXPECTED_LABEL_SIZES.values()) == EXPECTED_LABEL_BYTES

listing_result = run_gcloud([
    'storage', 'objects', 'list', f'{GCS_ROOT}/raw/**',
    '--format=json(name,size)',
])
listed_objects = json.loads(listing_result.stdout or '[]')

def object_basename(item):
    return str(item['name']).replace('\\', '/').rsplit('/', 1)[-1]

source_items = [item for item in listed_objects if object_basename(item).startswith('TS_') and object_basename(item).endswith('.zip')]
label_items = [item for item in listed_objects if object_basename(item).startswith('TL_') and object_basename(item).endswith('.zip')]
source_names = [object_basename(item) for item in source_items]
label_names = [object_basename(item) for item in label_items]
if len(source_names) != len(set(source_names)) or len(label_names) != len(set(label_names)):
    raise IOError('서로 다른 GCS 경로에 같은 ZIP 파일명이 중복되어 있습니다.')
source_sizes = {object_basename(item): int(item['size']) for item in source_items}
label_sizes = {object_basename(item): int(item['size']) for item in label_items}
source_keys = {Path(name).stem.removeprefix('TS_') for name in source_sizes}
label_keys = {Path(name).stem.removeprefix('TL_') for name in label_sizes}

validation_like = [item for item in listed_objects if object_basename(item).startswith(('VS_', 'VL_')) and object_basename(item).endswith('.zip')]
if validation_like:
    print('WARNING: 예선 분석 대상이 아닌 공식 Validation ZIP이 GCS raw 아래에 있습니다:')
    for item in validation_like[:10]:
        print(' -', item['name'])

print(f'Source ZIPs: {len(source_items)}/22')
print(f'Label ZIPs: {len(label_items)}/22')
print('Missing source:', sorted(EXPECTED_KEYS - source_keys))
print('Missing label:', sorted(EXPECTED_KEYS - label_keys))
print('Unexpected source:', sorted(source_keys - EXPECTED_KEYS))
print('Unexpected label:', sorted(label_keys - EXPECTED_KEYS))

wrong_source_sizes = {
    name: {'actual': source_sizes[name], 'expected': EXPECTED_SOURCE_SIZES[name]}
    for name in sorted(set(source_sizes) & set(EXPECTED_SOURCE_SIZES))
    if source_sizes[name] != EXPECTED_SOURCE_SIZES[name]
}
wrong_label_sizes = {
    name: {'actual': label_sizes[name], 'expected': EXPECTED_LABEL_SIZES[name]}
    for name in sorted(set(label_sizes) & set(EXPECTED_LABEL_SIZES))
    if label_sizes[name] != EXPECTED_LABEL_SIZES[name]
}
print('Wrong source sizes:', wrong_source_sizes)
print('Wrong label sizes:', wrong_label_sizes)
source_bytes = sum(source_sizes.values())
label_bytes = sum(label_sizes.values())
print(f'Source bytes: {source_bytes:,} / {EXPECTED_SOURCE_BYTES:,}')
print(f'Label bytes: {label_bytes:,} / {EXPECTED_LABEL_BYTES:,}')
keys_complete = source_keys == EXPECTED_KEYS and label_keys == EXPECTED_KEYS
sizes_complete = (
    source_sizes == EXPECTED_SOURCE_SIZES
    and label_sizes == EXPECTED_LABEL_SIZES
    and source_bytes == EXPECTED_SOURCE_BYTES
    and label_bytes == EXPECTED_LABEL_BYTES
)
if keys_complete and not sizes_complete:
    raise IOError('ZIP 이름은 모두 있지만 하나 이상의 exact byte 크기가 원본과 다릅니다.')
all_uploaded = keys_complete and sizes_complete
if all_uploaded:
    total_bytes = source_bytes + label_bytes
    print(f'Full Training upload audit: PASS ({total_bytes/1024**3:.3f} GiB)')
else:
    print('Full Training upload audit: NOT YET — 위 Missing 목록의 ZIP을 더 올리세요.')


## 완료 판정

다음이 모두 확인되면 환경·업로드 1차 준비가 끝난 것입니다.

- 런타임 정보 셀에서 원하는 경우 CUDA available: True와 GPU smoke PASS
- GCS에서 TS_H_0007.zip, TL_H_0007.zip 발견
- ZIP structure, pair set, and CRC: PASS
- Safe extraction: PASS
- Content validation: PASS
- GCS의 dcc/notebook/smoke_outputs에 CSV, PNG, 검증 JSON 생성
- 전체 업로드 후 Full Training upload audit: PASS

그 다음 단계는 전체 22개 묶음의 메타데이터 인덱스 구축, Group 기반 train/validation 분할, crop 생성, 모델 baseline 순서입니다. 검증 분할은 공식 Validation을 쓰지 않고 Training 내부에서 누수 없이 직접 만듭니다.

참고: https://research.google.com/colaboratory/faq.html
